# Main Notebook

* We need to decide on the portion of the video that is seen and that which is unseen - the unseen part, along with the annotations corresponding to them, will be that which is predicted on.
* We will start off by taking 10 seconds (60 frames) and then predicting the next 30
* We all still have to experiment with the 10-30 scheme we have going on - consequently, what we have are hyperparameters
* We will also try a 5-30, and a 10-60 scheme as well
* Then we will also try to reverse the general idea and have a long seen portion and predict a short unseen portion of the video
* We could also explore the idea of a causal transformer with teacher forcing in training, so that we can explore questions like "given what I have seen in the past, what will come next?"; but such a model is probably out of scope since it no longer matches the original idea
* All of these configurations will alter how we construct our examples, and therefore our batches

# 1. Imports, Mounting, Annotations, and WandB

In [ ]:
# Imports

import os
import json
import copy
import shutil
import collections

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import average_precision_score

In [ ]:
# Weights and Biases

import wandb
wandb.login()

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 2703800 (models-university-of-witwatersrand) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
# File paths

ROOT = "./HD-EPIC"
FEATURES = f"{ROOT}/Features"

Mounted at /content/drive


In [ ]:
# Loading in annotations from the HD-EPIC dataset itself

with open(f"{ROOT}/Annotations/scene-and-object-movements/assoc_info.json") as f:
    assoc_info = json.load(f)

with open(f"{ROOT}/Annotations/scene-and-object-movements/mask_info.json") as f:
    mask_info = json.load(f)

with open(f"{ROOT}/Annotations/eye-gaze-priming/priming_info.json") as f:
    priming_info = json.load(f)


noun_classes = pd.read_csv(
    f"{ROOT}/Annotations/narrations-and-action-segments/HD_EPIC_noun_classes.csv"
)

The annotations are in different file formats (JSON and CSV for example) - something, therefore, will need to be done to transform them into a usable state; or we could even use them to create features that contain the information that we need from them.

# 2. Configuration for task

In [ ]:
# Experiment configuration

CONTEXT_SECONDS = 30
PREDICTION_HORIZON = 30.0

FEATURE_INTERVAL_SECONDS = 1.0
SEQUENCE_LENGTH = round(CONTEXT_SECONDS / FEATURE_INTERVAL_SECONDS) - 1

FEATURE_DIM = 768

SEED = 42

CONFIG_NAME = f"context{CONTEXT_SECONDS}_horizon{int(PREDICTION_HORIZON)}_v2"

assert CONTEXT_SECONDS > 0
assert PREDICTION_HORIZON > 0
assert FEATURE_INTERVAL_SECONDS > 0

print(f"Configuration: {CONFIG_NAME}")
print(f"Context: {CONTEXT_SECONDS} seconds")
print(f"Prediction horizon: {PREDICTION_HORIZON} seconds")
print(f"Sequence length: {SEQUENCE_LENGTH} feature vectors")

Configuration: context30_horizon30_v2
Context: 30 seconds
Prediction horizon: 30.0 seconds
Sequence length: 29 feature vectors


In [ ]:
# Label creation functions

import ast

instance_to_class = {}

for _, row in noun_classes.iterrows():
    class_id = int(row["id"])
    for instance_name in ast.literal_eval(row["instances"]):
        if instance_name in instance_to_class:
            raise ValueError(f"Instance appears in multiple classes: {instance_name!r}")
        instance_to_class[instance_name] = class_id

# Add an unknown class for object names absent from the noun-class mapping.
NUM_KNOWN_CLASSES = len(noun_classes)
UNKNOWN_CLASS_ID = NUM_KNOWN_CLASSES
NUM_CLASSES = NUM_KNOWN_CLASSES + 1

def get_class_id(object_name):
    return instance_to_class.get(object_name, UNKNOWN_CLASS_ID)

def create_labels(video_annotations, clip_end_times, video_duration, horizon=PREDICTION_HORIZON,):
    """
    Create future object-relevance labels for one video.

    Parameters
    ----------
    video_annotations:
        Object annotations for one video from assoc_info.json.

    clip_end_times:
        Actual end timestamps of the extracted feature clips.

    video_duration:
        Duration of the video in seconds.

    horizon:
        Future prediction horizon in seconds.

    Returns
    -------
    Dictionary containing:
        observation_times
        relevance
        time_to_relevance
        time_mask
    """

    # Convert the input timestamps to a NumPy array.
    clip_end_times = np.asarray(clip_end_times, dtype=np.float64,)

    # Retain only observations for which the entire future
    # prediction horizon is covered by the video.
    valid_indices = np.flatnonzero(clip_end_times + horizon <= video_duration)

    observation_times = clip_end_times[valid_indices]

    # Allocate one target row per valid observation and
    # one column per output category.
    num_observations = len(observation_times)

    relevance = np.zeros((num_observations, NUM_CLASSES), dtype=np.float32,)

    time_to_relevance = np.zeros((num_observations, NUM_CLASSES), dtype=np.float32,)

    time_mask = np.zeros((num_observations, NUM_CLASSES), dtype=np.float32,)

    # Store the earliest future onset found for each
    # observation and class.
    earliest_onsets = np.full((num_observations, NUM_CLASSES), np.inf, dtype=np.float64,)

    # Examine every annotated object in this video.
    for object_id, object_info in video_annotations.items():

        # Map the object to its known class, or to unknown.
        object_name = object_info["name"]
        class_id = get_class_id(object_name)

        # Examine every temporal track belonging to the object.
        for track in object_info.get("tracks", []):

            # The first element is the track onset.
            track_start = float(track["time_segment"][0])

            # Find observations for which this onset lies
            # strictly in the future and within the horizon.
            delays = track_start - observation_times

            qualifies = ((delays > 0.0) & (delays <= horizon))

            # Find the observation rows affected by this onset.
            affected_rows = np.flatnonzero(qualifies)

            # Retain the earliest onset for each row and class.
            earliest_onsets[affected_rows, class_id] = np.minimum(earliest_onsets[affected_rows, class_id], track_start,)

    # Convert the earliest-onset information into targets.
    for observation_index, observation_time in enumerate(observation_times):

        # A finite value indicates that a qualifying future
        # track onset exists for this class.
        positive_classes = np.flatnonzero(np.isfinite(earliest_onsets[observation_index]))

        for class_id in positive_classes:
            onset_time = earliest_onsets[observation_index, class_id,]

            relevance[observation_index, class_id] = 1.0

            time_to_relevance[observation_index, class_id, ] = onset_time - observation_time

            time_mask[observation_index, class_id,] = 1.0

    return {
        "valid_indices": valid_indices,
        "observation_times": observation_times,
        "relevance": relevance,
        "time_to_relevance": time_to_relevance,
        "time_mask": time_mask,
    }

# 3. Loading features

In [ ]:
# Load the feature vectors for the dataset, which is in the next cell


features_dir = "./HD-EPIC/Dataset/videos"

MANIFEST_PATH = f"./HD-EPIC/Dataset/split_manifest.json"

os.makedirs(features_dir, exist_ok=True)

with open(MANIFEST_PATH, "r") as f:
    split_manifest = json.load(f)

required_splits = {"train", "validation", "test"}

assert set(split_manifest.keys()) == required_splits, (
    f"Expected exactly these splits: {required_splits}"
)

split_ids = {split: set(split_manifest[split]) for split in required_splits}

assert split_ids["train"].isdisjoint(split_ids["validation"])
assert split_ids["train"].isdisjoint(split_ids["test"])
assert split_ids["validation"].isdisjoint(split_ids["test"])

for split in ["train", "validation", "test"]:
    split_dir = os.path.join(features_dir, split)
    os.makedirs(split_dir, exist_ok=True)

    for video_id in sorted(split_ids[split]):
        filename = f"{video_id}.npz"
        source = os.path.join(drive_features_dir, filename)
        destination = os.path.join(split_dir, filename)

        if not os.path.isfile(source):
            raise FileNotFoundError(f"Missing feature file: {source}")

        if os.path.exists(destination) and os.path.getsize(source) == os.path.getsize(destination):
            continue

        print(f"Copying: {split}/{filename}")
        shutil.copy2(source, destination)

print("Feature files copied according to the existing split manifest.")

Copying: train/P07-20240529-102652.npz
Copying: train/P07-20240529-134410.npz
Copying: train/P07-20240530-102543.npz
Copying: train/P07-20240530-111344.npz
Copying: train/P07-20240531-201703.npz
Copying: train/P08-20240613-122900.npz
Copying: train/P08-20240613-123547.npz
Copying: train/P08-20240614-085000.npz
Copying: train/P08-20240614-125650.npz
Copying: train/P08-20240614-132938.npz
Copying: train/P08-20240614-184945.npz
Copying: train/P08-20240617-130401.npz
Copying: train/P08-20240617-184909.npz
Copying: train/P08-20240618-171546.npz
Copying: train/P08-20240620-180825.npz
Copying: train/P08-20240622-141902.npz
Copying: train/P09-20240621-153208.npz
Copying: train/P09-20240622-154652.npz
Copying: train/P09-20240622-162302.npz
Copying: train/P09-20240622-194642.npz
Copying: train/P09-20240623-153253.npz
Copying: train/P09-20240623-153748.npz
Copying: train/P09-20240624-165332.npz
Copying: validation/P07-20240529-191007.npz
Copying: validation/P07-20240529-194518.npz
Copying: valida

# 4. Dataset creation

In [ ]:
# Dataset


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

ROOT = "./HD-EPIC"
DATASET_DIR = f"{ROOT}/Dataset"
FEATURES_DIR = f"{DATASET_DIR}/videos"
MANIFEST_PATH = f"{DATASET_DIR}/split_manifest.json"

# Features should be sampled approximately once per second.
TIMESTAMP_TOLERANCE = 0.05

BATCH_SIZE = 32

assert os.path.isdir(FEATURES_DIR), f"Missing directory: {FEATURES_DIR}"
assert os.path.isfile(MANIFEST_PATH), f"Missing split manifest: {MANIFEST_PATH}"


# ------------------------------------------------------------
# Load and validate the existing split manifest
# ------------------------------------------------------------

with open(MANIFEST_PATH, "r") as f:
    split_manifest = json.load(f)

required_splits = {"train", "validation", "test"}

assert set(split_manifest.keys()) == required_splits, (
    f"Expected exactly these splits: {required_splits}"
)

split_ids = {}

for split_name in required_splits:
    split_ids[split_name] = list(split_manifest[split_name])

    assert len(split_ids[split_name]) == len(set(split_ids[split_name])), (
        f"Duplicate video IDs in {split_name} split."
    )

train_ids = set(split_ids["train"])
validation_ids = set(split_ids["validation"])
test_ids = set(split_ids["test"])

assert train_ids.isdisjoint(validation_ids), "Train/validation overlap."
assert train_ids.isdisjoint(test_ids), "Train/test overlap."
assert validation_ids.isdisjoint(test_ids), "Validation/test overlap."

print("Existing split manifest loaded.")
print(f"Training videos:   {len(train_ids)}")
print(f"Validation videos: {len(validation_ids)}")
print(f"Test videos:       {len(test_ids)}")


# ------------------------------------------------------------
# Find saved video datasets
# ------------------------------------------------------------

available_files = sorted(
    filename
    for filename in os.listdir(FEATURES_DIR)
    if filename.endswith(".npz")
)

available_ids = {
    os.path.splitext(filename)[0]
    for filename in available_files
}

manifest_ids = train_ids | validation_ids | test_ids

missing_ids = manifest_ids - available_ids
extra_ids = available_ids - manifest_ids

assert not missing_ids, f"Videos in manifest but missing files: {sorted(missing_ids)}"

if extra_ids:
    print(f"Warning: {len(extra_ids)} saved videos are absent from the manifest.")
    print("These videos will not be used.")

print(f"\nSaved video files: {len(available_ids)}")
print(f"Videos in manifest: {len(manifest_ids)}")


# ------------------------------------------------------------
# Load and validate every video dataset
# ------------------------------------------------------------

video_data = {}

total_observations = 0
total_positive_pairs = 0

required_keys = {"features", "observation_times", "video_duration"}
for video_id in sorted(manifest_ids):
    file_path = os.path.join(FEATURES_DIR, f"{video_id}.npz")

    with np.load(file_path) as data:
        assert required_keys.issubset(data.files), (
            f"{video_id}: missing keys {required_keys - set(data.files)}"
        )
        features = data["features"].astype(np.float32, copy=True)
        observation_times = data["observation_times"].astype(np.float32, copy=True)
        video_duration = float(data["video_duration"])

    # generate labels for the current config
    labels = create_labels(assoc_info[video_id], observation_times, video_duration,
                          horizon=PREDICTION_HORIZON)
    features = features[labels["valid_indices"]]
    observation_times = labels["observation_times"].astype(np.float32)
    relevance = labels["relevance"]
    time_to_relevance = labels["time_to_relevance"]
    time_mask = labels["time_mask"]

    n = len(observation_times)

    # All arrays must agree on the number of observations.
    assert features.shape == (n, FEATURE_DIM), (
        f"{video_id}: unexpected feature shape {features.shape}"
    )

    assert relevance.shape == (n, NUM_CLASSES), (
        f"{video_id}: unexpected relevance shape {relevance.shape}"
    )

    assert time_to_relevance.shape == (n, NUM_CLASSES), (
        f"{video_id}: unexpected time-to-relevance shape {time_to_relevance.shape}"
    )

    assert time_mask.shape == (n, NUM_CLASSES), (
        f"{video_id}: unexpected time-mask shape {time_mask.shape}"
    )

    assert np.all(np.isfinite(features)), f"{video_id}: non-finite features."
    assert np.all(np.isfinite(observation_times)), f"{video_id}: non-finite timestamps."
    assert np.all(np.diff(observation_times) > 0), (
        f"{video_id}: timestamps are not strictly increasing."
    )

    assert np.all((relevance == 0) | (relevance == 1)), (
        f"{video_id}: relevance contains values other than 0 and 1."
    )

    assert np.all((time_mask == 0) | (time_mask == 1)), (
        f"{video_id}: time_mask contains values other than 0 and 1."
    )

    # Under our current label definition, time targets are valid exactly
    # where relevance is positive.
    assert np.array_equal(relevance, time_mask), (
        f"{video_id}: relevance and time_mask disagree."
    )

    positive_mask = time_mask == 1
    negative_mask = time_mask == 0

    assert np.all(time_to_relevance[positive_mask] > 0), (
        f"{video_id}: positive target has a non-positive time."
    )

    assert np.all(time_to_relevance[positive_mask] <= PREDICTION_HORIZON), (
        f"{video_id}: positive target exceeds the prediction horizon."
    )

    assert np.all(time_to_relevance[negative_mask] == 0), (
        f"{video_id}: invalid time targets should be zero."
    )

    # The labels should only describe events within the complete future horizon.
    assert np.all(
        observation_times + PREDICTION_HORIZON <= video_duration + 1e-3
    ), f"{video_id}: an observation lacks a complete future horizon."

    video_data[video_id] = {
        "features": features,
        "observation_times": observation_times,
        "relevance": relevance,
        "time_to_relevance": time_to_relevance,
        "time_mask": time_mask,
        "video_duration": video_duration
    }

    total_observations += n
    total_positive_pairs += int(relevance.sum())

print("\nAll saved video datasets passed validation.")
print(f"Videos loaded: {len(video_data)}")
print(f"Observations: {total_observations:,}")
print(f"Positive observation/class pairs: {total_positive_pairs:,}")


# ------------------------------------------------------------
# Build sequences of recent feature vectors
# ------------------------------------------------------------

def find_valid_endpoints(observation_times, sequence_length=SEQUENCE_LENGTH):
    """
    Return indices that can serve as endpoints of valid input sequences.

    Each input consists of sequence_length consecutive feature vectors.
    Consecutive observation timestamps must be approximately one second apart.
    """
    valid_endpoints = []

    for end_index in range(sequence_length - 1, len(observation_times)):
        start_index = end_index - sequence_length + 1

        sequence_times = observation_times[start_index:end_index + 1]
        intervals = np.diff(sequence_times)

        # Reject sequences containing missing or irregularly spaced observations.
        if np.all(np.abs(intervals - FEATURE_INTERVAL_SECONDS) <= TIMESTAMP_TOLERANCE):
            valid_endpoints.append(end_index)

    return valid_endpoints


valid_endpoints_by_video = {}
skipped_windows_by_video = {}

for video_id, arrays in video_data.items():
    n = len(arrays["observation_times"])

    valid_endpoints = find_valid_endpoints(arrays["observation_times"])

    valid_endpoints_by_video[video_id] = valid_endpoints
    skipped_windows_by_video[video_id] = max(
        0, n - SEQUENCE_LENGTH + 1
    ) - len(valid_endpoints)

print("\nSequence construction complete.")
print(f"Required input length: {SEQUENCE_LENGTH} feature vectors")
print(f"Feature dimension: {FEATURE_DIM}")
print(f"Expected timestamp interval: {FEATURE_INTERVAL_SECONDS:.1f} second")


# ------------------------------------------------------------
# 6. Create PyTorch Dataset
# ------------------------------------------------------------

class EpicDataset(Dataset):
    def __init__(self, video_ids, video_data, valid_endpoints_by_video):
        self.video_data = video_data
        self.examples = []

        for video_id in sorted(video_ids):
            for end_index in valid_endpoints_by_video[video_id]:
                self.examples.append((video_id, end_index))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        video_id, end_index = self.examples[index]
        arrays = self.video_data[video_id]

        start_index = end_index - SEQUENCE_LENGTH + 1

        # Shape: [10, 768]
        features = arrays["features"][start_index:end_index + 1]

        # Targets refer to the final observation in the sequence.
        relevance = arrays["relevance"][end_index]
        time_to_relevance = arrays["time_to_relevance"][end_index]
        time_mask = arrays["time_mask"][end_index]
        observation_time = arrays["observation_times"][end_index]

        return {
            "features": torch.from_numpy(features.copy()),
            "relevance": torch.from_numpy(relevance.copy()),
            "time_to_relevance": torch.from_numpy(time_to_relevance.copy()),
            "time_mask": torch.from_numpy(time_mask.copy()),
            "observation_time": torch.tensor(observation_time, dtype=torch.float32),
            "video_id": video_id
        }


# ------------------------------------------------------------
# Create train, validation, and test datasets
# ------------------------------------------------------------

train_dataset = EpicDataset(train_ids, video_data, valid_endpoints_by_video)
validation_dataset = EpicDataset(validation_ids, video_data, valid_endpoints_by_video)
test_dataset = EpicDataset(test_ids, video_data, valid_endpoints_by_video)

print("\nFinal dataset sizes:")
print(f"Training examples:   {len(train_dataset):,}")
print(f"Validation examples: {len(validation_dataset):,}")
print(f"Test examples:       {len(test_dataset):,}")
print(
    "Total examples:      "
    f"{len(train_dataset) + len(validation_dataset) + len(test_dataset):,}"
)


# ------------------------------------------------------------
# Summarise each split
# ------------------------------------------------------------

def summarise_split(split_name, video_ids, dataset):
    observations = sum(len(video_data[v]["observation_times"]) for v in video_ids)
    positive_pairs = sum(
        int(video_data[v]["relevance"].sum()) for v in video_ids
    )
    videos_with_examples = sum(
        len(valid_endpoints_by_video[v]) > 0 for v in video_ids
    )

    print(f"\n{split_name.upper()}")
    print(f"Videos: {len(video_ids)}")
    print(f"Videos contributing examples: {videos_with_examples}")
    print(f"Saved observations: {observations:,}")
    print(f"Positive observation/class pairs: {positive_pairs:,}")
    print(f"Usable sequence examples: {len(dataset):,}")


summarise_split("train", train_ids, train_dataset)
summarise_split("validation", validation_ids, validation_dataset)
summarise_split("test", test_ids, test_dataset)

Existing split manifest loaded.
Training videos:   23
Validation videos: 6
Test videos:       5

Saved video files: 34
Videos in manifest: 34

All saved video datasets passed validation.
Videos loaded: 34
Observations: 43,334
Positive observation/class pairs: 125,683

Sequence construction complete.
Required input length: 29 feature vectors
Feature dimension: 768
Expected timestamp interval: 1.0 second

Final dataset sizes:
Training examples:   22,808
Validation examples: 14,509
Test examples:       5,089
Total examples:      42,406

TRAIN
Videos: 23
Videos contributing examples: 23
Saved observations: 23,452
Positive observation/class pairs: 74,604
Usable sequence examples: 22,808

VALIDATION
Videos: 6
Videos contributing examples: 6
Saved observations: 14,677
Positive observation/class pairs: 38,591
Usable sequence examples: 14,509

TEST
Videos: 5
Videos contributing examples: 4
Saved observations: 5,205
Positive observation/class pairs: 12,488
Usable sequence examples: 5,089


# 5. Dataloader creation

In [ ]:
# Dataloaders - we will define the batch sizes here

generator = torch.Generator()
generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    generator=generator
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

# 6. Model creation

In [ ]:
# Model

import random
import torch.nn.functional as F

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Confirm the dataset configuration
assert FEATURE_DIM == 768

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


class TemporalTransformer(nn.Module):
    def __init__(self, feature_dim=768, hidden_dim=256, num_heads=4, num_layers=4, sequence_length, num_classes=304, prediction_horizon=30.0, dropout=0.1):
        super().__init__()

        self.sequence_length = sequence_length
        self.num_classes = num_classes
        self.prediction_horizon = prediction_horizon

        # Project the VideoMAE features into a smaller representation.
        self.input_projection = nn.Linear(feature_dim, hidden_dim)

        # Give each position in the sequence a learned temporal embedding.
        self.positional_embedding = nn.Parameter(torch.randn(1, sequence_length, hidden_dim) * 0.02)

        # Process the temporal sequence.
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
            activation="gelu"
        )

        self.temporal_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Predict whether each class will become relevant.
        self.relevance_head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

        # Predict the time until each class's next qualifying onset.
        self.timing_head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, features):
        # Input: [batch_size, 11, 768]
        expected_shape = (self.sequence_length, self.input_projection.in_features)

        if features.ndim != 3 or tuple(features.shape[1:]) != expected_shape:
            raise ValueError(f"Expected [batch, {expected_shape[0]}, {expected_shape[1]}], got {tuple(features.shape)}")

        x = self.input_projection(features)
        x = x + self.positional_embedding

        # Unrestricted self-attention across the observed sequence.
        x = self.temporal_encoder(x)

        # Use the representation at the latest observation time.
        latest_representation = x[:, -1, :]

        relevance_logits = self.relevance_head(latest_representation)

        # Bound timing predictions to the 30-second horizon.
        time_predictions = torch.sigmoid(self.timing_head(latest_representation)) * self.prediction_horizon

        return {
            "relevance_logits": relevance_logits,
            "time_predictions": time_predictions
        }

Device: cuda


# 7. Weight calculation

In [ ]:
# Calculating weights

positive_counts = torch.zeros(NUM_CLASSES)
total_training_examples = 0

for batch in train_loader:
    labels = batch["relevance"]
    positive_counts += labels.sum(dim=0)
    total_training_examples += labels.shape[0]

negative_counts = total_training_examples - positive_counts

pos_weight = (negative_counts / positive_counts.clamp_min(1)).clamp(max=20.0)

# Classes with no positive training examples receive the default weight.
pos_weight[positive_counts == 0] = 1.0

zero_positive_classes = (positive_counts == 0).sum().item()

print(f"Training examples: {total_training_examples:,}")
print(f"Classes with positive training examples: {(positive_counts > 0).sum().item()}/{NUM_CLASSES}")
print(f"Classes with no positive training examples: {zero_positive_classes}/{NUM_CLASSES}")
print(f"Positive-class weight range: {pos_weight.min().item():.2f}–{pos_weight.max().item():.2f}")

Training examples: 22,808
Classes with positive training examples: 105/304
Classes with no positive training examples: 199/304
Positive-class weight range: 0.60–20.00


# 8. Loss functions

In [ ]:
# Loss function - calculates both losses

class FutureRelevanceLoss(nn.Module):
    def __init__(self, pos_weight):
        super().__init__()
        self.register_buffer("pos_weight", pos_weight)

    def forward(self, outputs, batch):
        # Binary relevance classification.
        relevance_loss = F.binary_cross_entropy_with_logits(
            outputs["relevance_logits"],
            batch["relevance"],
            pos_weight=self.pos_weight
        )

        # Timing regression, using only valid targets.
        valid = batch["time_mask"].bool()

        if valid.any():
            timing_errors = F.smooth_l1_loss(
                outputs["time_predictions"][valid],
                batch["time_to_relevance"][valid],
                beta=1.0,
                reduction="sum"
            )

            timing_loss = timing_errors / valid.sum()
        else:
            # Return a differentiable zero if this batch has no valid targets.
            timing_loss = outputs["time_predictions"].sum() * 0.0

        total_loss = relevance_loss + timing_loss

        return {
            "total": total_loss,
            "relevance": relevance_loss,
            "timing": timing_loss
        }

# 9. Instances

In [ ]:
# Instance of our model, optimiser, and loss function

model = TemporalTransformer(
    feature_dim=FEATURE_DIM,
    hidden_dim=256,
    num_heads=4,
    num_layers=4,
    sequence_length=SEQUENCE_LENGTH,
    num_classes=NUM_CLASSES,
    prediction_horizon=PREDICTION_HORIZON,
    dropout=0.1
).to(device)

loss_fn = FutureRelevanceLoss(pos_weight.to(device))

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

num_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model parameters: {num_parameters:,}")
print(f"Training device: {device}")

/tmp/ipykernel_3005/2972094466.py:47: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.temporal_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


Model parameters: 3,520,608
Training device: cuda


# 10. Training and Validating the model

In [ ]:
# Training the model and recording metrics on Weights & Biases

def run_epoch(loader, training):
    model.train(training)

    relevance_loss_total = 0.0
    timing_loss_total = 0.0
    valid_timing_count = 0
    example_count = 0

    with torch.set_grad_enabled(training):
        for batch in loader:
            features = batch["features"].to(device)

            targets = {
                "relevance": batch["relevance"].to(device),
                "time_to_relevance": batch["time_to_relevance"].to(device),
                "time_mask": batch["time_mask"].to(device)
            }

            if training:
                optimizer.zero_grad(set_to_none=True)

            outputs = model(features)
            losses = loss_fn(outputs, targets)

            if training:
                losses["total"].backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            batch_size = features.shape[0]
            example_count += batch_size

            relevance_loss_total += losses["relevance"].item() * batch_size

            valid = targets["time_mask"].bool()
            if valid.any():
                timing_errors = F.smooth_l1_loss(
                    outputs["time_predictions"][valid],
                    targets["time_to_relevance"][valid],
                    beta=1.0,
                    reduction="sum"
                )
                timing_loss_total += timing_errors.item()
                valid_timing_count += valid.sum().item()

    if example_count == 0:
        raise ValueError("The data loader contains no examples.")

    relevance_loss = relevance_loss_total / example_count
    timing_loss = timing_loss_total / max(valid_timing_count, 1)

    return {
        "total": relevance_loss + timing_loss,
        "relevance": relevance_loss,
        "timing": timing_loss
    }


# Training configuration

CHECKPOINT_PATH = f"{DATASET_DIR}/best_{CONFIG_NAME}.pt"
os.makedirs(DATASET_DIR, exist_ok=True)

NUM_EPOCHS = 30

best_val_loss = float("inf")
best_epoch = 0

param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

config = {
    "architecture": "TemporalTransformer",
    "optimizer": optimizer.__class__.__name__,
    "learning_rate": optimizer.param_groups[0]["lr"],
    "weight_decay": optimizer.param_groups[0]["weight_decay"],
    "epochs": NUM_EPOCHS,
    "batch_size": train_loader.batch_size,
    "parameters": num_parameters,
    "seed": SEED,
    "feature_dim": FEATURE_DIM,
    "hidden_dim": 256,
    "num_heads": 4,
    "num_layers": 4,
    "sequence_length": SEQUENCE_LENGTH,
    "num_classes": NUM_CLASSES,
    "prediction_horizon": PREDICTION_HORIZON,
    "context_seconds": CONTEXT_SECONDS,
    "config_name" : CONFIG_NAME,
    "dropout": 0.1
}

run_name = f"transformer_{CONFIG_NAME}"

run = wandb.init(project="cv-project", name=run_name, config=config)
run_id = run.id

try:
    for epoch in range(1, NUM_EPOCHS + 1):
        train_metrics = run_epoch(train_loader, training=True)
        val_metrics = run_epoch(validation_loader, training=False)

        print(
            f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
            f"Train loss: {train_metrics['total']:.4f} "
            f"(relevance: {train_metrics['relevance']:.4f}, timing: {train_metrics['timing']:.4f}) | "
            f"Validation loss: {val_metrics['total']:.4f} "
            f"(relevance: {val_metrics['relevance']:.4f}, timing: {val_metrics['timing']:.4f})"
        )

        if val_metrics["total"] < best_val_loss:
            best_val_loss = val_metrics["total"]
            best_epoch = epoch
            epochs_without_improvement = 0

            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "validation_loss": best_val_loss,
                "config": config
            }, CHECKPOINT_PATH)

            print("  Saved new best checkpoint.")

        else:
            epochs_without_improvement += 1

        # Record metrics for this epoch on W&B.
        wandb.log({
            "epoch": epoch,
            "train/total_loss": train_metrics["total"],
            "train/relevance_loss": train_metrics["relevance"],
            "train/timing_loss": train_metrics["timing"],
            "validation/total_loss": val_metrics["total"],
            "validation/relevance_loss": val_metrics["relevance"],
            "validation/timing_loss": val_metrics["timing"],
            "best/validation_loss": best_val_loss,
            "best/epoch": best_epoch,
            "learning_rate": optimizer.param_groups[0]["lr"]
        }, step=epoch)

finally:
    wandb.finish()

print(f"\nBest epoch: {best_epoch}")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Checkpoint: {CHECKPOINT_PATH}")

Epoch 01/30 | Train loss: 6.9795 (relevance: 0.2691, timing: 6.7104) | Validation loss: 7.1622 (relevance: 0.1892, timing: 6.9730)
  Saved new best checkpoint.
Epoch 02/30 | Train loss: 5.7216 (relevance: 0.2039, timing: 5.5177) | Validation loss: 7.5133 (relevance: 0.1956, timing: 7.3177)
Epoch 03/30 | Train loss: 4.5152 (relevance: 0.1853, timing: 4.3299) | Validation loss: 7.7337 (relevance: 0.1891, timing: 7.5446)
Epoch 04/30 | Train loss: 3.6612 (relevance: 0.1678, timing: 3.4934) | Validation loss: 7.8395 (relevance: 0.1859, timing: 7.6537)
Epoch 05/30 | Train loss: 3.1314 (relevance: 0.1539, timing: 2.9774) | Validation loss: 7.9853 (relevance: 0.1868, timing: 7.7985)
Epoch 06/30 | Train loss: 2.7931 (relevance: 0.1428, timing: 2.6503) | Validation loss: 8.0298 (relevance: 0.1899, timing: 7.8399)
Epoch 07/30 | Train loss: 2.5660 (relevance: 0.1338, timing: 2.4322) | Validation loss: 7.7817 (relevance: 0.1956, timing: 7.5862)
Epoch 08/30 | Train loss: 2.3761 (relevance: 0.1261, t

best/epoch,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best/validation_loss,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/relevance_loss,█▆▅▄▄▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
train/timing_loss,█▇▅▄▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
train/total_loss,█▆▅▄▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
validation/relevance_loss,▁▂▁▁▁▁▂▂▂▂▃▃▃▄▄▅▅▅▅▅▆▆▆▇▇▇████
validation/timing_loss,▁▄▆▆██▆▇▆▇▇▆▆▇▇▆▆▆▆▆▆▆▅▅▅▆▅▅▅▅
validation/total_loss,▁▄▆▆██▆▇▇▇▇▆▇▇▇▆▇▆▆▇▇▆▆▆▆▆▆▆▆▆
best/epoch,1



Best epoch: 1
Best validation loss: 7.1622
Checkpoint: /content/drive/MyDrive/Project/HD-EPIC/Dataset_v1/best_context30_horizon30_v2.pt


In [ ]:
# Reloading the best model generated in training, since the last epoch does not necessarily give us the best

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"Loaded best model from epoch {checkpoint['epoch']}.")
print(f"Best validation loss: {checkpoint['validation_loss']:.4f}")

Loaded best model from epoch 1.
Best validation loss: 7.1622


In [ ]:
# Evaluation of the model on the validation set - need to record this on WandB

from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score

def collect_predictions(model, loader, device):
    all_probabilities = []
    all_relevance = []
    all_predicted_times = []
    all_actual_times = []
    all_time_masks = []

    model.eval()

    with torch.inference_mode():
        for batch in loader:
            features = batch["features"].to(device)
            outputs = model(features)

            probabilities = torch.sigmoid(outputs["relevance_logits"])

            all_probabilities.append(probabilities.cpu())
            all_relevance.append(batch["relevance"].cpu())
            all_predicted_times.append(outputs["time_predictions"].cpu())
            all_actual_times.append(batch["time_to_relevance"].cpu())
            all_time_masks.append(batch["time_mask"].bool().cpu())

    return {
        "probabilities": torch.cat(all_probabilities).numpy(),
        "relevance": torch.cat(all_relevance).numpy(),
        "predicted_times": torch.cat(all_predicted_times).numpy(),
        "actual_times": torch.cat(all_actual_times).numpy(),
        "time_masks": torch.cat(all_time_masks).numpy()
    }


val_results = collect_predictions(model, validation_loader, device)

probabilities = val_results["probabilities"]
relevance = val_results["relevance"]
predicted_times = val_results["predicted_times"]
actual_times = val_results["actual_times"]
time_masks = val_results["time_masks"]

print("\nValidation predictions collected.")
print(f"Observations: {len(relevance):,}")
print(f"Classes per observation: {relevance.shape[1]}")
print(f"Relevance predictions shape: {probabilities.shape}")
print(f"Timing predictions shape: {predicted_times.shape}")
print(f"Valid timing targets: {time_masks.sum():,}")

predicted_labels = (probabilities >= 0.5).astype(np.int32)
true_labels = relevance.astype(np.int32)

micro_ap = average_precision_score(true_labels.ravel(), probabilities.ravel())

classes_with_positives = true_labels.sum(axis=0) > 0
macro_ap = average_precision_score(true_labels[:, classes_with_positives],probabilities[:, classes_with_positives], average="macro")

micro_precision = precision_score(true_labels.ravel(), predicted_labels.ravel(), zero_division=0)
micro_recall = recall_score(true_labels.ravel(), predicted_labels.ravel(), zero_division=0)
micro_f1 = f1_score(true_labels.ravel(), predicted_labels.ravel(), zero_division=0)

print()
print()

print("RELEVANCE RESULTS")
print("-" * 45)
print(f"Micro average precision: {micro_ap:.4f}")
print(f"Macro average precision: {macro_ap:.4f}")
print(f"Micro precision at threshold 0.5: {micro_precision:.4f}")
print(f"Micro recall at threshold 0.5:    {micro_recall:.4f}")
print(f"Micro F1 at threshold 0.5:        {micro_f1:.4f}")
print(f"Classes with positive validation labels: {classes_with_positives.sum()}/{NUM_CLASSES}")

valid = time_masks

true_times = actual_times[valid]
model_times = predicted_times[valid]

absolute_errors = np.abs(model_times - true_times)

model_mae = absolute_errors.mean()
model_median_error = np.median(absolute_errors)
model_smooth_l1 = F.smooth_l1_loss(
    torch.from_numpy(model_times),
    torch.from_numpy(true_times),
    beta=1.0
).item()

print()
print()

print("TIMING RESULTS")
print("-" * 45)
print(f"Valid timing targets: {len(true_times):,}")
print(f"Mean absolute error:   {model_mae:.3f} seconds")
print(f"Median absolute error: {model_median_error:.3f} seconds")
print(f"Smooth L1 loss:        {model_smooth_l1:.4f}")

training_times_by_class = [[] for _ in range(NUM_CLASSES)]

for batch in train_loader:
    times = batch["time_to_relevance"]
    masks = batch["time_mask"].bool()

    for class_id in range(NUM_CLASSES):
        class_times = times[:, class_id][masks[:, class_id]]
        training_times_by_class[class_id].extend(class_times.tolist())

all_training_times = [
    time
    for class_times in training_times_by_class
    for time in class_times
]

if not all_training_times:
    raise ValueError("No valid training timing targets were found.")

global_median_time = float(np.median(all_training_times))

class_median_times = np.array([
    np.median(class_times) if class_times else global_median_time
    for class_times in training_times_by_class
])

baseline_times = np.broadcast_to(class_median_times, predicted_times.shape)

baseline_errors = np.abs(baseline_times[valid] - true_times)
baseline_mae = baseline_errors.mean()
baseline_median_error = np.median(baseline_errors)

baseline_smooth_l1 = F.smooth_l1_loss(torch.from_numpy(baseline_times[valid]), torch.from_numpy(true_times), beta=1.0).item()

print()
print()

print("TIMING BASELINE COMPARISON")
print("-" * 55)
print(f"{'Metric':<25}{'Transformer':>15}{'Baseline':>15}")
print("-" * 55)
print(f"{'MAE (seconds)':<25}{model_mae:>15.3f}{baseline_mae:>15.3f}")
print(f"{'Median error (seconds)':<25}{model_median_error:>15.3f}{baseline_median_error:>15.3f}")
print(f"{'Smooth L1 loss':<25}{model_smooth_l1:>15.4f}{baseline_smooth_l1:>15.4f}")

training_positive_counts = torch.zeros(NUM_CLASSES)
training_example_count = 0

for batch in train_loader:
    labels = batch["relevance"]
    training_positive_counts += labels.sum(dim=0)
    training_example_count += labels.shape[0]

class_prevalence = (training_positive_counts / training_example_count).numpy()

baseline_probabilities = np.broadcast_to(class_prevalence, probabilities.shape)

baseline_micro_ap = average_precision_score(
    true_labels.ravel(),
    baseline_probabilities.ravel())

baseline_macro_ap = average_precision_score(true_labels[:, classes_with_positives], baseline_probabilities[:, classes_with_positives], average="macro")

print()
print()

print("RELEVANCE BASELINE COMPARISON")
print("-" * 45)
print(f"{'Metric':<25}{'Transformer':>12}{'Baseline':>12}")
print("-" * 49)
print(f"{'Micro AP':<25}{micro_ap:>12.4f}{baseline_micro_ap:>12.4f}")
print(f"{'Macro AP':<25}{macro_ap:>12.4f}{baseline_macro_ap:>12.4f}")

print()
print()

print("=" * 65)
print("VALIDATION EVALUATION SUMMARY")
print("=" * 65)

print("\nRelevance:")
print(f"  Transformer micro AP: {micro_ap:.4f}")
print(f"  Baseline micro AP:    {baseline_micro_ap:.4f}")
print(f"  Transformer macro AP: {macro_ap:.4f}")
print(f"  Baseline macro AP:    {baseline_macro_ap:.4f}")

print("\nTiming:")
print(f"  Transformer MAE:      {model_mae:.3f} seconds")
print(f"  Baseline MAE:         {baseline_mae:.3f} seconds")
print(f"  Transformer Smooth L1: {model_smooth_l1:.4f}")
print(f"  Baseline Smooth L1:    {baseline_smooth_l1:.4f}")

print("\nCheckpoint:")
print(f"  Epoch:                {checkpoint['epoch']}")
print(f"  Validation loss:      {checkpoint['validation_loss']:.4f}")

run = wandb.init(project="cv-project", id=run_id, resume="must")

run.summary.update({
    "val/micro_ap": micro_ap,
    "val/macro_ap": macro_ap,
    "val/micro_f1": micro_f1,
    "val/timing_mae": model_mae,
    "val/timing_median_error": model_median_error,
    "val/baseline_micro_ap": baseline_micro_ap,
    "val/baseline_macro_ap": baseline_macro_ap,
    "val/baseline_timing_mae": baseline_mae,
    "val/best_epoch": checkpoint["epoch"],
})
run.finish()


Validation predictions collected.
Observations: 14,509
Classes per observation: 304
Relevance predictions shape: (14509, 304)
Timing predictions shape: (14509, 304)
Valid timing targets: 37,961


RELEVANCE RESULTS
---------------------------------------------
Micro average precision: 0.1137
Macro average precision: 0.0466
Micro precision at threshold 0.5: 0.1227
Micro recall at threshold 0.5:    0.4319
Micro F1 at threshold 0.5:        0.1911
Classes with positive validation labels: 83/304


TIMING RESULTS
---------------------------------------------
Valid timing targets: 37,961
Mean absolute error:   7.461 seconds
Median absolute error: 6.831 seconds
Smooth L1 loss:        6.9730


TIMING BASELINE COMPARISON
-------------------------------------------------------
Metric                       Transformer       Baseline
-------------------------------------------------------
MAE (seconds)                      7.461          7.299
Median error (seconds)             6.831          6.965

best/epoch,1
best/validation_loss,7.16219
epoch,30
learning_rate,0.0001
train/relevance_loss,0.07036
train/timing_loss,1.08871
train/total_loss,1.15906
val/baseline_macro_ap,0.03152
val/baseline_micro_ap,0.25724
val/baseline_timing_mae,7.2991
+9,...


# 11. Testing the model

In [ ]:
# Evaluation of the model on testing set - need to log this also

# Load checkpoint model
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score

CHECKPOINT_PATH = f"{DATASET_DIR}/best_{CONFIG_NAME}.pt"

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)

model = TemporalTransformer(
    feature_dim=FEATURE_DIM,
    hidden_dim=256,
    num_heads=4,
    num_layers=4,
    sequence_length=SEQUENCE_LENGTH,
    num_classes=NUM_CLASSES,
    prediction_horizon=PREDICTION_HORIZON,
    dropout=0.1
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")
print(f"Best validation loss: {checkpoint['validation_loss']:.4f}")
print(f"Evaluation device: {device}")

print()
print()

# Collect predictions from test set
def collect_predictions(model, loader, device):
    all_probabilities = []
    all_relevance = []
    all_predicted_times = []
    all_actual_times = []
    all_time_masks = []

    model.eval()

    with torch.inference_mode():
        for batch in loader:
            features = batch["features"].to(device)
            outputs = model(features)

            all_probabilities.append(torch.sigmoid(outputs["relevance_logits"]).cpu())
            all_relevance.append(batch["relevance"].cpu())
            all_predicted_times.append(outputs["time_predictions"].cpu())
            all_actual_times.append(batch["time_to_relevance"].cpu())
            all_time_masks.append(batch["time_mask"].bool().cpu())

    if not all_relevance:
        raise ValueError("The test loader contains no examples.")

    return {
        "probabilities": torch.cat(all_probabilities).numpy(),
        "relevance": torch.cat(all_relevance).numpy(),
        "predicted_times": torch.cat(all_predicted_times).numpy(),
        "actual_times": torch.cat(all_actual_times).numpy(),
        "time_masks": torch.cat(all_time_masks).numpy()
    }


test_results = collect_predictions(model, test_loader, device)

probabilities = test_results["probabilities"]
relevance = test_results["relevance"]
predicted_times = test_results["predicted_times"]
actual_times = test_results["actual_times"]
time_masks = test_results["time_masks"]

print(f"Test observations: {len(relevance):,}")
print(f"Relevance predictions: {probabilities.shape}")
print(f"Timing predictions: {predicted_times.shape}")
print(f"Valid timing targets: {time_masks.sum():,}")

print()
print()

# Evaluate relevance predictions
predicted_labels = (probabilities >= 0.5).astype(np.int32)
true_labels = relevance.astype(np.int32)

micro_ap = average_precision_score(true_labels.ravel(), probabilities.ravel())

classes_with_positives = true_labels.sum(axis=0) > 0
macro_ap = average_precision_score(
    true_labels[:, classes_with_positives],
    probabilities[:, classes_with_positives],
    average="macro"
)

micro_precision = precision_score(true_labels.ravel(), predicted_labels.ravel(), zero_division=0)
micro_recall = recall_score(true_labels.ravel(), predicted_labels.ravel(), zero_division=0)
micro_f1 = f1_score(true_labels.ravel(), predicted_labels.ravel(), zero_division=0)

print("\nTEST RELEVANCE RESULTS")
print("-" * 45)
print(f"Micro average precision:       {micro_ap:.4f}")
print(f"Macro average precision:       {macro_ap:.4f}")
print(f"Micro precision at threshold:  {micro_precision:.4f}")
print(f"Micro recall at threshold:     {micro_recall:.4f}")
print(f"Micro F1 at threshold:         {micro_f1:.4f}")
print(f"Classes with positive labels:  {classes_with_positives.sum()}/{NUM_CLASSES}")

print()
print()

# Evaluate timing predictions
valid = time_masks

true_times = actual_times[valid]
model_times = predicted_times[valid]

if len(true_times) == 0:
    raise ValueError("The test set contains no valid timing targets.")

absolute_errors = np.abs(model_times - true_times)

model_mae = absolute_errors.mean()
model_median_error = np.median(absolute_errors)
model_smooth_l1 = F.smooth_l1_loss(
    torch.from_numpy(model_times),
    torch.from_numpy(true_times),
    beta=1.0
).item()

print("\nTEST TIMING RESULTS")
print("-" * 45)
print(f"Valid timing targets: {len(true_times):,}")
print(f"Mean absolute error:   {model_mae:.3f} seconds")
print(f"Median absolute error: {model_median_error:.3f} seconds")
print(f"Smooth L1 loss:        {model_smooth_l1:.4f}")

print()
print()

# Evaluate timing baseline
training_times_by_class = [[] for _ in range(NUM_CLASSES)]

for batch in train_loader:
    times = batch["time_to_relevance"]
    masks = batch["time_mask"].bool()

    for class_id in range(NUM_CLASSES):
        class_times = times[:, class_id][masks[:, class_id]]
        training_times_by_class[class_id].extend(class_times.tolist())

all_training_times = [time for class_times in training_times_by_class for time in class_times]

if not all_training_times:
    raise ValueError("No valid training timing targets were found.")

global_median_time = float(np.median(all_training_times))

class_median_times = np.array([
    np.median(class_times) if class_times else global_median_time
    for class_times in training_times_by_class
])

baseline_times = np.broadcast_to(class_median_times, predicted_times.shape)

baseline_errors = np.abs(baseline_times[valid] - true_times)
baseline_mae = baseline_errors.mean()
baseline_median_error = np.median(baseline_errors)
baseline_smooth_l1 = F.smooth_l1_loss(
    torch.from_numpy(baseline_times[valid]),
    torch.from_numpy(true_times),
    beta=1.0
).item()

print("\nTEST TIMING BASELINE COMPARISON")
print("-" * 55)
print(f"{'Metric':<25}{'Transformer':>15}{'Baseline':>15}")
print("-" * 55)
print(f"{'MAE (seconds)':<25}{model_mae:>15.3f}{baseline_mae:>15.3f}")
print(f"{'Median error (seconds)':<25}{model_median_error:>15.3f}{baseline_median_error:>15.3f}")
print(f"{'Smooth L1 loss':<25}{model_smooth_l1:>15.4f}{baseline_smooth_l1:>15.4f}")

print()
print()

# Evaluate relevance baseline
training_positive_counts = torch.zeros(NUM_CLASSES)
training_example_count = 0

for batch in train_loader:
    labels = batch["relevance"]
    training_positive_counts += labels.sum(dim=0)
    training_example_count += labels.shape[0]

class_prevalence = (training_positive_counts / training_example_count).numpy()
baseline_probabilities = np.broadcast_to(class_prevalence, probabilities.shape)

baseline_micro_ap = average_precision_score(
    true_labels.ravel(),
    baseline_probabilities.ravel()
)

baseline_macro_ap = average_precision_score(
    true_labels[:, classes_with_positives],
    baseline_probabilities[:, classes_with_positives],
    average="macro"
)

print("\nTEST RELEVANCE BASELINE COMPARISON")
print("-" * 50)
print(f"{'Metric':<25}{'Transformer':>12}{'Baseline':>12}")
print("-" * 49)
print(f"{'Micro AP':<25}{micro_ap:>12.4f}{baseline_micro_ap:>12.4f}")
print(f"{'Macro AP':<25}{macro_ap:>12.4f}{baseline_macro_ap:>12.4f}")

print()
print()

# Final summary
print("\n" + "=" * 65)
print("FINAL TEST EVALUATION SUMMARY")
print("=" * 65)

print("\nRelevance:")
print(f"  Transformer micro AP: {micro_ap:.4f}")
print(f"  Baseline micro AP:    {baseline_micro_ap:.4f}")
print(f"  Transformer macro AP: {macro_ap:.4f}")
print(f"  Baseline macro AP:    {baseline_macro_ap:.4f}")
print(f"  Transformer F1:       {micro_f1:.4f}")

print("\nTiming:")
print(f"  Transformer MAE:      {model_mae:.3f} seconds")
print(f"  Baseline MAE:         {baseline_mae:.3f} seconds")
print(f"  Transformer Smooth L1: {model_smooth_l1:.4f}")
print(f"  Baseline Smooth L1:   {baseline_smooth_l1:.4f}")

print("\nCheckpoint:")
print(f"  Selected epoch:       {checkpoint['epoch']}")
print(f"  Best validation loss: {checkpoint['validation_loss']:.4f}")

run = wandb.init(project="cv-project", id=run_id, resume="must")

run.summary.update({
    "test/micro_ap": micro_ap,
    "test/macro_ap": macro_ap,
    "test/micro_f1": micro_f1,
    "test/timing_mae": model_mae,
    "test/timing_median_error": model_median_error,
    "test/baseline_micro_ap": baseline_micro_ap,
    "test/baseline_macro_ap": baseline_macro_ap,
    "test/baseline_timing_mae": baseline_mae,
    "test/best_epoch": checkpoint["epoch"],
})
run.finish()

/tmp/ipykernel_3005/2972094466.py:47: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.temporal_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


Loaded checkpoint from epoch 1
Best validation loss: 7.1622
Evaluation device: cuda


Test observations: 5,089
Relevance predictions: (5089, 304)
Timing predictions: (5089, 304)
Valid timing targets: 11,881



TEST RELEVANCE RESULTS
---------------------------------------------
Micro average precision:       0.1015
Macro average precision:       0.0701
Micro precision at threshold:  0.0980
Micro recall at threshold:     0.4448
Micro F1 at threshold:         0.1606
Classes with positive labels:  60/304



TEST TIMING RESULTS
---------------------------------------------
Valid timing targets: 11,881
Mean absolute error:   7.664 seconds
Median absolute error: 7.032 seconds
Smooth L1 loss:        7.1755



TEST TIMING BASELINE COMPARISON
-------------------------------------------------------
Metric                       Transformer       Baseline
-------------------------------------------------------
MAE (seconds)                      7.664          7.389
Median error (seconds)          

best/epoch,1
best/validation_loss,7.16219
epoch,30
learning_rate,0.0001
test/baseline_macro_ap,0.03891
test/baseline_micro_ap,0.20998
test/baseline_timing_mae,7.38942
test/best_epoch,1
test/macro_ap,0.07007
test/micro_ap,0.10151
+18,...


**Weights and Biases Links**

Here, we will past the links to the runs.

# Dataset Reference

```bibtex
@InProceedings{perrett2025hdepic,
  author    = {Perrett, Toby and Darkhalil, Ahmad and Sinha, Saptarshi and Emara, Omar and Pollard, Sam and Parida, Kranti and Liu, Kaiting and Gatti, Prajwal and Bansal, Siddhant and Flanagan, Kevin and Chalk, Jacob and Zhu, Zhifan and Guerrier, Rhodri and Abdelazim, Fahd and Zhu, Bin and Moltisanti, Davide and Wray, Michael and Doughty, Hazel and Damen, Dima},
  title     = {HD-EPIC: A Highly-Detailed Egocentric Video Dataset},
  booktitle = {Proceedings of the IEEE/CVF Conference on Computer Vision and Pattern Recognition (CVPR)},
  year      = {2025},
  month     = {June}
}
```